# SRD Multi-Band — SmolVLM-256M Visual Benchmark

Tests four correction modes on TextVQA (visual exact-match accuracy):

| Mode | What's corrected |
|---|---|
| `baseline` | No correction — pure Q4 fake-quant |
| `lm_only` | Language backbone reasoning layers (40–77%) |
| `connector_lm` | Cross-modal projector + LM layers |
| `all_bands` | Vision encoder + connector + LM (full multi-band) |

**Runtime:** T4 GPU recommended. ~25 min for 100 questions, ~2 hr for full 5k.

In [ ]:
# ── CELL 1: GPU check + mount Drive ──────────────────────────────────────
import torch
from google.colab import drive

p = torch.cuda.get_device_properties(0)
print(f"GPU: {p.name}  {p.total_memory/1024**3:.1f} GB VRAM")

drive.mount("/content/drive")

from pathlib import Path
OUT_DIR = Path("/content/drive/MyDrive/srd_output")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("✓  Drive mounted")

In [ ]:
# ── CELL 2: Clone axiom repo + install deps ───────────────────────────────
import subprocess, sys
from pathlib import Path

REPO = Path("/content/axiom")

if not REPO.is_dir():
    subprocess.run([
        "git", "clone", "--depth", "1",
        "--branch", "claude/srd-multimodal",
        "https://github.com/orivael-dev/axiom.git",
        str(REPO)
    ], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "pull"], check=True)

# Pin transformers to 4.44.2 — transformers 5.x removed AutoModelForVision2Seq
# which SmolVLM (Idefics3) requires. Do NOT use --upgrade or >=4.40 here.
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.44.2", "accelerate", "datasets",
    "sentencepiece", "Pillow"
], check=True)

sys.path.insert(0, str(REPO))
print("✓  Setup complete")

In [ ]:
# ── CELL 3: Dry run — detect SmolVLM components, no inference ────────────
from research.quant.bench_multimodal_srd import run_benchmark

run_benchmark(dry_run=True)

In [ ]:
# ── CELL 4: Run benchmark — 100 questions (~25 min on T4) ─────────────────
# Each of the 4 modes loads SmolVLM fresh, applies correction, runs 100 VQA questions.
# Results are appended to Drive so you can resume if the session drops.

from research.quant.bench_multimodal_srd import run_benchmark, _print_summary
from pathlib import Path

results = run_benchmark(
    n_questions=100,
    output_path=Path("/content/drive/MyDrive/srd_output/multimodal_results.json"),
)
_print_summary(results)

In [ ]:
# ── CELL 5 (optional): Full validation set — 5000 questions (~2 hr) ───────
# Uncomment and run only if you want statistically clean results.
# Use A100 or T4 High-RAM for this cell.

# results_full = run_benchmark(
#     n_questions=5000,
#     output_path=Path("/content/drive/MyDrive/srd_output/multimodal_results_full.json"),
# )
# _print_summary(results_full)

In [ ]:
# ── CELL 6: Load + display saved results (resume after session drop) ───────
import json
from pathlib import Path
from research.quant.bench_multimodal_srd import _print_summary

results_path = Path("/content/drive/MyDrive/srd_output/multimodal_results.json")
if results_path.exists():
    saved = json.loads(results_path.read_text())
    _print_summary(saved)
else:
    print("No results file yet — run Cell 4 first")

In [ ]:
# ── CELL 7: Write ruleset sidecars for all GGUFs on Drive ─────────────────
# Run after building GGUFs to generate .axiom_ruleset.json for each model.

from research.quant.axiom_rulesets import write_all_rulesets
from pathlib import Path

write_all_rulesets(Path("/content/drive/MyDrive/srd_output"))

In [ ]:
# ── CELL 8: Print Gemma3 ruleset (example) ────────────────────────────────
import json
from research.quant.axiom_rulesets import RULESETS

print(json.dumps(RULESETS["gemma3-1b"], indent=2))